In [4]:
import importlib

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os

import model_functions
import utils

importlib.reload(model_functions)
importlib.reload(utils)

from model_functions import load_transition_matrix, gen_risk_matrix, gen_transition_probabilities, run_markov_cohort, \
    run_mc_sim, calculate_outcomes, plot_trace, load_costs, STAGE_ORDER, run_comparison, run_mc_sim_complex

In [5]:
STAGE_ORDER = ['F0', 'F1', 'F2', 'F3', 'F4', 'HCC', 'DC', 'LT', 'post-LT', 'Death']
# TODO Add in 3_mo_post_LT, 6_mo_post_LT, 12_mo_post_LT??

MORTALITY_COLUMNS = ['Death']
# TODO split Death into liver-related, all-cause, cardiovascular?

DIR_HRP = "./" # TODO global variable

In [6]:
cycle_len = 1/13
n_cycles = 88 * 13
cohort_size = 1000  # 5
tm = load_transition_matrix('./final_baseline_transition_probs.csv', "_obs", cycle_length=cycle_len)
rm = gen_risk_matrix(regression_risk=1.56, progression_risk=0.6097424116233234)

initial_prevalence = [.628, 0.309, 0.035, 0.016, 0.012, 0,0,0,0,0]

# treatment starting at 12, 72 weeks of treatment
mc_trace_12_72, drug_trace_12_72 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=18, drug_stages=['F2', 'F3'])

In [7]:
# treatment starting at 18, 72 weeks of treatment
mc_trace_18_72, drug_trace_18_72 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=6*13, tx_cycle_num=18, drug_stages=['F2', 'F3'])

In [8]:
((mc_trace_12_72 != 9).sum(axis=1) / 13 + 12).describe()

count    1000.000000
mean       44.709077
std        14.792447
min        12.461538
25%        32.826923
50%        45.115385
75%        56.538462
max        78.615385
dtype: float64

In [9]:
((mc_trace_18_72 != 9).sum(axis=1) / 13 + 12).describe()

count    1000.000000
mean       44.055308
std        14.364609
min        12.230769
25%        32.846154
50%        44.500000
75%        55.230769
max        80.846154
dtype: float64

In [10]:
(drug_trace_12_72.sum(axis=1)).describe()

count    1000.000000
mean       10.887000
std         8.678222
min         0.000000
25%         0.000000
50%        18.000000
75%        18.000000
max        18.000000
dtype: float64

In [11]:
drug_trace_18_72.sum(axis=1).describe()

count    1000.00000
mean       10.49600
std         8.75995
min         0.00000
25%         0.00000
50%        18.00000
75%        18.00000
max        18.00000
dtype: float64

In [12]:
import numpy as np
import pandas as pd

def convert_to_cohort_trace(markov_trace, stage_order, age_index=None):
    """
    Converts an individual-level Markov trace to a cohort-level trace.
    
    Parameters:
    - markov_trace: 2D numpy array or Pandas DataFrame (n_individuals, n_cycles) 
                    containing the integer state indices.
    - stage_order: List of state names (strings) corresponding to the indices.
    - age_index: Optional array, list, or pandas Index to use as the row labels (ages).
    
    Returns:
    - pd.DataFrame: Cohort trace (rows = age, columns = stages, values = proportions)
    """
    # Ensure we are working with a fast numpy integer array
    trace_array = np.asarray(markov_trace).astype(int)
    n_individuals, n_cols = trace_array.shape
    n_states = len(stage_order)
    
    # Pre-allocate an empty matrix for the results
    cohort_matrix = np.zeros((n_cols, n_states))
    
    # Iterate through each cycle (column) and count the states
    for t in range(n_cols):
        # bincount tallies occurrences of each state index (0, 1, 2...).
        # minlength ensures states with 0 patients are still represented.
        state_counts = np.bincount(trace_array[:, t], minlength=n_states)
        
        # Convert to proportions and store in our matrix
        cohort_matrix[t, :] = state_counts / n_individuals
        
    # Convert back to a nicely formatted Pandas DataFrame
    cohort_df = pd.DataFrame(cohort_matrix, columns=stage_order)
    cohort_df[['age_float', 'age']] = markov_trace.columns.tolist()
        
    return cohort_df

In [13]:
mc_test = convert_to_cohort_trace(mc_trace_18_72, STAGE_ORDER)
oc_18_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_72)

Total LY:  32.01684615384615
Total QALY:  27.449610962
Total Cost:  502805.6268516989
Discounted LY:  19.461993835326204
Discounted QALY:  16.98473151968566
Discounted Cost:  285007.7209936332


In [14]:
mc_test = convert_to_cohort_trace(mc_trace_12_72, STAGE_ORDER)
oc_12_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_72)

Total LY:  32.67061538461539
Total QALY:  27.952054400846155
Total Cost:  515281.05390002095
Discounted LY:  19.66835001744499
Discounted QALY:  17.13727517143973
Discounted Cost:  289368.9506567483


In [50]:
mc_test = convert_to_cohort_trace(mc_trace_18_72, STAGE_ORDER)
oc_18_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_72)

Total LY:  32.32484615384615
Total QALY:  27.671604628846154
Total Cost:  523681.2036712792
Discounted LY:  19.481514990524147
Discounted QALY:  16.98135774067161
Discounted Cost:  291337.0900525188


In [48]:
mc_test = convert_to_cohort_trace(mc_trace_12_72, STAGE_ORDER)
oc_12_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_72)

Total LY:  32.05615384615385
Total QALY:  27.504569317384618
Total Cost:  504271.92632670264
Discounted LY:  19.413960714133555
Discounted QALY:  16.95511705911038
Discounted Cost:  283380.24147572723


In [3]:
import numpy as np
import pandas as pd
import os
import concurrent.futures
import multiprocessing

# --- Worker Function for Parallel Processing ---
# This function runs a "chunk" of the total iterations on a single core.
def _sim_worker(args):
    (n_iter_chunk, base_transition_matrix, init_state, n_cycles, starting_age, 
     cycle_length, risk_matrix, begin_tx_at_cycle, tx_cycle_num, drug_stages, overall_mortality) = args

    rng = np.random.default_rng()
    
    # We assume STAGE_ORDER is defined globally in the module
    markov_trace = np.zeros((n_iter_chunk, n_cycles + 1))
    on_drug = np.zeros((n_iter_chunk, n_cycles + 1))
    num_drug_cycles = np.zeros(n_iter_chunk)

    for i in range(n_iter_chunk):
        starting_state = rng.choice(len(init_state), p=init_state)
        markov_trace[i, 0] = starting_state
        
        for t in range(n_cycles):
            age = starting_age + t * cycle_length
            
            local_risk_matrix = risk_matrix.copy() if risk_matrix is not None else None
            this_stage_idx = int(markov_trace[i, t])
            this_stage = STAGE_ORDER[this_stage_idx]

            if this_stage in MORTALITY_COLUMNS:
                markov_trace[i, t + 1:] = this_stage_idx
                break
            
            local_on_drug = (((drug_stages is not None and this_stage in drug_stages) or
                              (drug_stages is None)) and
                             begin_tx_at_cycle <= t and
                             num_drug_cycles[i] < tx_cycle_num)
            
            if local_on_drug:
                on_drug[i, t] = 1
                num_drug_cycles[i] += 1
            else:
                local_risk_matrix = None

            t_matrix = gen_transition_probabilities(
                base_transition_matrix.copy(), 
                np.floor(age),
                overall_mortality,  # Passed from memory, not disk
                local_risk_matrix
            )
            
            markov_trace[i, t + 1] = rng.choice(
                len(init_state), 
                p=t_matrix.iloc[this_stage_idx, :].values
            )

    return markov_trace, on_drug


# --- Main Parallelized Function ---
def run_mc_sim_complex(base_transition_matrix, init_state, n_cycles, n_iter, 
                       starting_age=12, cycle_length=1., risk_matrix=None, 
                       begin_tx_at_cycle=0, tx_cycle_num=None, drug_stages=None,
                       n_jobs=-1):
    
    assert len(init_state) == len(STAGE_ORDER), "Initial state vector length must match number of states."
    assert len(init_state) == len(base_transition_matrix), "Transition matrix size must match number of states."

    if tx_cycle_num is None:
        tx_cycle_num = n_cycles

    # Update gen_transition_probabilities to accept this object directly.
    overall_mortality = pd.read_csv(os.path.join(DIR_HRP, "background_mortality.csv"), index_col=0) 

    # 2. DETERMINE CPU CORES & CHUNKING STRATEGY
    if n_jobs == -1:
        n_jobs = multiprocessing.cpu_count() # Uses all 8 cores on your VM
    
    # Split the n_iter into chunks for each core to process
    base_chunk = n_iter // n_jobs
    chunks = [base_chunk] * n_jobs
    for i in range(n_iter % n_jobs):
        chunks[i] += 1 # Distribute remainder

    # Create arguments for each worker
    worker_args = [
        (chunk, base_transition_matrix, init_state, n_cycles, starting_age, 
         cycle_length, risk_matrix, begin_tx_at_cycle, tx_cycle_num, drug_stages, overall_mortality)
        for chunk in chunks if chunk > 0
    ]

    # 3. RUN IN PARALLEL
    all_markov_traces = []
    all_on_drugs = []
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        results = executor.map(_sim_worker, worker_args)
        
        for trace_chunk, drug_chunk in results:
            all_markov_traces.append(trace_chunk)
            all_on_drugs.append(drug_chunk)

    # Recombine results from all cores
    final_markov_trace = np.vstack(all_markov_traces)
    final_on_drug = np.vstack(all_on_drugs)

    # 4. FORMAT DATAFRAMES
    trace_df = pd.DataFrame(final_markov_trace).T
    trace_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    trace_df['age'] = np.floor(trace_df['age_float'])
    trace_df.set_index(['age_float', 'age'], inplace=True)

    on_drug_df = pd.DataFrame(final_on_drug).T
    on_drug_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    on_drug_df['age'] = np.floor(on_drug_df['age_float'])
    on_drug_df.set_index(['age_float', 'age'], inplace=True)

    return trace_df.T, on_drug_df.T

# Longer treatment 

In [15]:

# treatment starting at 12, 260 weeks of treatment
mc_trace_12_260, drug_trace_12_260 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=13*5, drug_stages=['F2', 'F3'])
# treatment starting at 18, 260 weeks of treatment
mc_trace_18_260, drug_trace_18_260 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=13*6, tx_cycle_num=13*5, drug_stages=['F2', 'F3'])

In [16]:
mc_test = convert_to_cohort_trace(mc_trace_12_260, STAGE_ORDER)
oc_12_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_260)

Total LY:  32.836615384615385
Total QALY:  28.15287495973077
Total Cost:  535808.09744538
Discounted LY:  19.71400275427033
Discounted QALY:  17.210567196283716
Discounted Cost:  298661.9669782791


In [17]:
mc_test = convert_to_cohort_trace(mc_trace_18_260, STAGE_ORDER)
oc_18_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_260)

Total LY:  33.11746153846154
Total QALY:  28.36656333753846
Total Cost:  537700.0944158158
Discounted LY:  19.89419435592912
Discounted QALY:  17.353910978460323
Discounted Cost:  299507.3254249287


In [52]:
mc_test = convert_to_cohort_trace(mc_trace_12_260, STAGE_ORDER)
oc_12_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_260)

Total LY:  32.129769230769234
Total QALY:  27.599605091461537
Total Cost:  512166.0594650397
Discounted LY:  19.47172542772258
Discounted QALY:  17.01699447484606
Discounted Cost:  290615.0486551501


In [53]:
mc_test = convert_to_cohort_trace(mc_trace_18_260, STAGE_ORDER)
oc_18_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_260)

Total LY:  32.76423076923077
Total QALY:  28.080246407884616
Total Cost:  531743.8440875582
Discounted LY:  19.77327868887688
Discounted QALY:  17.25332028365611
Discounted Cost:  295784.4128130932
